In [ ]:
# ============================================================
# PT -> PA Cross-Physics Mapping
# Wavelet Neural Operator (WNO2D)
#
# Input:
#     PT [B, 1, 501, 200]
#
# Output:
#     PA [B, 1, 501, 200]
#
# Core idea:
#
# x
#  -> lifting
#  -> Haar DWT
#  -> learnable wavelet-domain operator
#  -> Haar IDWT
#  -> residual/local mixing
#  -> projection
#
# Fully differentiable PyTorch implementation.
#
# No external wavelet package is required.
# ============================================================


# ============================================================
# Imports
# ============================================================

import os
import csv
import glob
import random

from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import h5py

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader


# ============================================================
# Config
# ============================================================

DATA_ROOT = "Training dataset"


RUN_TIME = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


OUT_DIR = os.path.join(
    "wno2d_mat_dataset_results",
    f"run_{RUN_TIME}"
)


os.makedirs(
    OUT_DIR,
    exist_ok=True
)


print("Results will be saved to:")
print(OUT_DIR)


# ============================================================
# Dataset split
# ============================================================

TRAIN_RATIO = 0.8

VAL_RATIO = 0.1

TEST_RATIO = 0.1

SEED = 42


# ============================================================
# Device
# ============================================================

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


print("Using device:", DEVICE)


# ============================================================
# Data dimensions
# ============================================================

NT = 501

NX = 200


# ============================================================
# Training parameters
# ============================================================

EPOCHS = 1000

BATCH_SIZE = 4

LR = 1e-3

WEIGHT_DECAY = 1e-5
# ============================================================
# Early stopping
# ============================================================


# ============================================================
# WNO parameters
# ============================================================

IN_CHANNELS = 1

OUT_CHANNELS = 1


# feature width
WIDTH = 48


# number of WNO blocks
N_LAYERS = 4


# number of Haar decomposition levels
#
# 2 is recommended first.
#
# 501 x 200
# is padded automatically.
#
# level 1:
# ~252 x 100
#
# level 2:
# ~126 x 50
#
WAVELET_LEVELS = 2


# ============================================================
# Random seed
# ============================================================

random.seed(
    SEED
)

np.random.seed(
    SEED
)

torch.manual_seed(
    SEED
)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(
        SEED
    )


# ============================================================
# Find MAT files
# ============================================================

all_files = glob.glob(

    os.path.join(
        DATA_ROOT,
        "**",
        "*.mat"
    ),

    recursive=True
)


all_files = sorted(
    all_files
)


print(
    "Total .mat samples found:",
    len(all_files)
)


assert len(all_files) > 0, \
    f"No .mat files found under: {DATA_ROOT}"


if len(all_files) != 1000:

    print(

        f"Warning: expected 1000 samples, "
        f"but found {len(all_files)}"
    )


# ============================================================
# Random dataset split
# ============================================================

random.shuffle(
    all_files
)


n_total = len(
    all_files
)


n_train = int(
    n_total * TRAIN_RATIO
)


n_val = int(
    n_total * VAL_RATIO
)


train_files = all_files[
    :n_train
]


val_files = all_files[
    n_train:
    n_train + n_val
]


test_files = all_files[
    n_train + n_val:
]


print("\nDataset split:")

print(
    "Train samples:",
    len(train_files)
)

print(
    "Val samples  :",
    len(val_files)
)

print(
    "Test samples :",
    len(test_files)
)


# ============================================================
# Compute normalization statistics
#
# TRAINING DATA ONLY
# ============================================================

def compute_statistics(
    file_list
):

    pt_sum = 0.0

    pt_sq_sum = 0.0


    pa_sum = 0.0

    pa_sq_sum = 0.0


    n_elements = 0


    print(
        "\nCalculating normalization statistics..."
    )


    for mat_path in tqdm(

        file_list,

        desc="Statistics",

        ncols=120
    ):


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(

                f["PT"],

                dtype=np.float64
            )


            PA = np.asarray(

                f["PA"],

                dtype=np.float64
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        pt_sum += PT.sum()


        pt_sq_sum += np.square(
            PT
        ).sum()


        pa_sum += PA.sum()


        pa_sq_sum += np.square(
            PA
        ).sum()


        n_elements += PT.size


    # ========================================================
    # Mean
    # ========================================================

    pt_mean = (

        pt_sum
        /
        n_elements
    )


    pa_mean = (

        pa_sum
        /
        n_elements
    )


    # ========================================================
    # Variance
    # ========================================================

    pt_var = (

        pt_sq_sum
        /
        n_elements

        -

        pt_mean ** 2
    )


    pa_var = (

        pa_sq_sum
        /
        n_elements

        -

        pa_mean ** 2
    )


    pt_var = max(
        pt_var,
        0.0
    )


    pa_var = max(
        pa_var,
        0.0
    )


    # ========================================================
    # Standard deviation
    # ========================================================

    pt_std = (

        np.sqrt(
            pt_var
        )

        +

        1e-8
    )


    pa_std = (

        np.sqrt(
            pa_var
        )

        +

        1e-8
    )


    return (

        pt_mean,
        pt_std,

        pa_mean,
        pa_std
    )


# ============================================================
# Normalization
# ============================================================

(
    pt_mean,
    pt_std,

    pa_mean,
    pa_std

) = compute_statistics(
    train_files
)


print(
    "\nNormalization statistics:"
)


print(
    f"PT mean = {pt_mean:.6e}"
)


print(
    f"PT std  = {pt_std:.6e}"
)


print(
    f"PA mean = {pa_mean:.6e}"
)


print(
    f"PA std  = {pa_std:.6e}"
)


# ============================================================
# Save normalization
# ============================================================

np.savez(

    os.path.join(
        OUT_DIR,
        "normalization_parameters.npz"
    ),

    pt_mean=pt_mean,

    pt_std=pt_std,

    pa_mean=pa_mean,

    pa_std=pa_std
)


# ============================================================
# Dataset
# ============================================================

class MatHeatAcousticDataset(Dataset):


    def __init__(
        self,
        file_list,
        pt_mean,
        pt_std,
        pa_mean,
        pa_std
    ):


        self.file_list = file_list


        self.pt_mean = pt_mean

        self.pt_std = pt_std


        self.pa_mean = pa_mean

        self.pa_std = pa_std


    def __len__(
        self
    ):

        return len(
            self.file_list
        )


    def __getitem__(
        self,
        idx
    ):


        mat_path = self.file_list[
            idx
        ]


        with h5py.File(
            mat_path,
            "r"
        ) as f:


            PT = np.asarray(

                f["PT"],

                dtype=np.float32
            )


            PA = np.asarray(

                f["PA"],

                dtype=np.float32
            )


        # ====================================================
        # Shape
        # ====================================================

        assert PT.shape == (NT, NX), \
            f"Wrong PT shape in {mat_path}: {PT.shape}"


        assert PA.shape == (NT, NX), \
            f"Wrong PA shape in {mat_path}: {PA.shape}"


        # ====================================================
        # Normalize
        # ====================================================

        PT = (

            PT
            -
            self.pt_mean

        ) / self.pt_std


        PA = (

            PA
            -
            self.pa_mean

        ) / self.pa_std


        # ====================================================
        # [T,X]
        #
        # ->
        #
        # [C,T,X]
        # ====================================================

        PT = torch.tensor(

            PT,

            dtype=torch.float32

        ).unsqueeze(
            0
        )


        PA = torch.tensor(

            PA,

            dtype=torch.float32

        ).unsqueeze(
            0
        )


        return (
            PT,
            PA
        )


    # ========================================================
    # Denormalization
    # ========================================================

    def denormalize_pa(
        self,
        x
    ):


        return (

            x
            *
            self.pa_std

            +

            self.pa_mean
        )


    def denormalize_pt(
        self,
        x
    ):


        return (

            x
            *
            self.pt_std

            +

            self.pt_mean
        )


# ============================================================
# Datasets
# ============================================================

train_dataset = MatHeatAcousticDataset(

    train_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


val_dataset = MatHeatAcousticDataset(

    val_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


test_dataset = MatHeatAcousticDataset(

    test_files,

    pt_mean,
    pt_std,

    pa_mean,
    pa_std
)


# ============================================================
# DataLoaders
# ============================================================

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


# ============================================================
# Dataset check
# ============================================================

PT_batch, PA_batch = next(

    iter(
        train_loader
    )
)


print(
    "\nBatch check:"
)


print(
    "PT batch shape:",
    PT_batch.shape
)


print(
    "PA batch shape:",
    PA_batch.shape
)


# ============================================================
# 2D Haar DWT
#
# Input:
#
# [B,C,H,W]
#
# Output:
#
# LL
# LH
# HL
# HH
#
# each:
#
# [B,C,H/2,W/2]
#
# ============================================================

class HaarDWT2D(nn.Module):


    def __init__(
        self
    ):


        super().__init__()


        # ====================================================
        # Orthonormal Haar filters
        # ====================================================

        s = 0.5


        LL = torch.tensor(
            [
                [s, s],
                [s, s]
            ],
            dtype=torch.float32
        )


        LH = torch.tensor(
            [
                [-s, -s],
                [ s,  s]
            ],
            dtype=torch.float32
        )


        HL = torch.tensor(
            [
                [-s, s],
                [-s, s]
            ],
            dtype=torch.float32
        )


        HH = torch.tensor(
            [
                [ s, -s],
                [-s,  s]
            ],
            dtype=torch.float32
        )


        filters = torch.stack(

            [
                LL,
                LH,
                HL,
                HH
            ],

            dim=0
        )


        # ====================================================
        # [4,1,2,2]
        # ====================================================

        filters = filters.unsqueeze(
            1
        )


        self.register_buffer(

            "filters",

            filters
        )


    def forward(
        self,
        x
    ):


        B, C, H, W = x.shape


        # ====================================================
        # Make dimensions even
        # ====================================================

        pad_h = H % 2

        pad_w = W % 2


        if (
            pad_h != 0
            or
            pad_w != 0
        ):


            x = F.pad(

                x,

                (
                    0,
                    pad_w,
                    0,
                    pad_h
                ),

                mode="replicate"
            )


        # ====================================================
        # Repeat filters for each channel
        # ====================================================

        filters = self.filters.repeat(

            C,

            1,

            1,

            1
        )


        # ====================================================
        # Group convolution
        # ====================================================

        y = F.conv2d(

            x,

            filters,

            stride=2,

            groups=C
        )


        # ====================================================
        # Current ordering:
        #
        # channel1:
        # LL LH HL HH
        #
        # channel2:
        # LL LH HL HH
        #
        # ...
        #
        # reshape:
        #
        # [B,C,4,H/2,W/2]
        # ====================================================

        H2 = y.shape[
            -2
        ]


        W2 = y.shape[
            -1
        ]


        y = y.reshape(

            B,

            C,

            4,

            H2,

            W2
        )


        LL = y[
            :,
            :,
            0
        ]


        LH = y[
            :,
            :,
            1
        ]


        HL = y[
            :,
            :,
            2
        ]


        HH = y[
            :,
            :,
            3
        ]


        return (

            LL,
            LH,
            HL,
            HH
        )


# ============================================================
# 2D Haar inverse DWT
# ============================================================

class HaarIDWT2D(nn.Module):


    def __init__(
        self
    ):


        super().__init__()


        s = 0.5


        LL = torch.tensor(
            [
                [s, s],
                [s, s]
            ],
            dtype=torch.float32
        )


        LH = torch.tensor(
            [
                [-s, -s],
                [ s,  s]
            ],
            dtype=torch.float32
        )


        HL = torch.tensor(
            [
                [-s, s],
                [-s, s]
            ],
            dtype=torch.float32
        )


        HH = torch.tensor(
            [
                [ s, -s],
                [-s,  s]
            ],
            dtype=torch.float32
        )


        filters = torch.stack(

            [
                LL,
                LH,
                HL,
                HH
            ],

            dim=0
        )


        filters = filters.unsqueeze(
            1
        )


        self.register_buffer(

            "filters",

            filters
        )


    def forward(
        self,
        LL,
        LH,
        HL,
        HH
    ):


        B, C, H, W = LL.shape


        # ====================================================
        # Stack wavelet bands
        # ====================================================

        y = torch.stack(

            [
                LL,
                LH,
                HL,
                HH
            ],

            dim=2
        )


        # ====================================================
        # [B,C,4,H,W]
        #
        # ->
        #
        # [B,4C,H,W]
        # ====================================================

        y = y.reshape(

            B,

            4 * C,

            H,

            W
        )


        filters = self.filters.repeat(

            C,

            1,

            1,

            1
        )


        # ====================================================
        # Transposed group convolution
        # ====================================================

        x = F.conv_transpose2d(

            y,

            filters,

            stride=2,

            groups=C
        )


        return x


# ============================================================
# Wavelet-domain mixing
#
# Each subband receives its own learnable operator.
#
# LL = low-frequency / approximation
#
# LH, HL, HH = localized high-frequency information
# ============================================================

class WaveletBandMix(nn.Module):


    def __init__(
        self,
        channels
    ):


        super().__init__()


        # ====================================================
        # Independent learnable operators
        # ====================================================

        self.ll = nn.Conv2d(

            channels,

            channels,

            kernel_size=1
        )


        self.lh = nn.Conv2d(

            channels,

            channels,

            kernel_size=1
        )


        self.hl = nn.Conv2d(

            channels,

            channels,

            kernel_size=1
        )


        self.hh = nn.Conv2d(

            channels,

            channels,

            kernel_size=1
        )


    def forward(
        self,
        LL,
        LH,
        HL,
        HH
    ):


        LL = self.ll(
            LL
        )


        LH = self.lh(
            LH
        )


        HL = self.hl(
            HL
        )


        HH = self.hh(
            HH
        )


        return (

            LL,
            LH,
            HL,
            HH
        )


# ============================================================
# One multi-level Wavelet Operator
# ============================================================

class WaveletOperator2D(nn.Module):


    def __init__(
        self,
        channels,
        levels=2
    ):


        super().__init__()


        self.levels = levels


        self.dwt = HaarDWT2D()

        self.idwt = HaarIDWT2D()


        # ====================================================
        # Learnable wavelet operators for every scale
        # ====================================================

        self.band_mix = nn.ModuleList(

            [

                WaveletBandMix(
                    channels
                )

                for _ in range(
                    levels
                )
            ]
        )


    def forward(
        self,
        x
    ):


        original_size = x.shape[
            -2:
        ]


        # ====================================================
        # Save all levels
        # ====================================================

        details = []

        sizes = []


        current = x


        # ====================================================
        # Forward multi-resolution decomposition
        # ====================================================

        for level in range(
            self.levels
        ):


            sizes.append(
                current.shape[-2:]
            )


            (
                LL,
                LH,
                HL,
                HH

            ) = self.dwt(
                current
            )


            (
                LL,
                LH,
                HL,
                HH

            ) = self.band_mix[
                level
            ](

                LL,
                LH,
                HL,
                HH
            )


            details.append(

                (
                    LH,
                    HL,
                    HH
                )
            )


            current = LL


        # ====================================================
        # current now contains the coarsest LL representation
        # ====================================================


        # ====================================================
        # Reconstruction
        # ====================================================

        for level in reversed(

            range(
                self.levels
            )
        ):


            (
                LH,
                HL,
                HH

            ) = details[
                level
            ]


            current = self.idwt(

                current,
                LH,
                HL,
                HH
            )


            # =================================================
            # Crop to size before this DWT level
            # =================================================

            target_h = sizes[
                level
            ][0]


            target_w = sizes[
                level
            ][1]


            current = current[
                :,
                :,
                :target_h,
                :target_w
            ]


        # ====================================================
        # Final safety crop
        # ====================================================

        current = current[
            :,
            :,
            :original_size[0],
            :original_size[1]
        ]


        return current


# ============================================================
# WNO Block
#
# Same general neural operator philosophy as FNO:
#
# wavelet operator branch
#
# +
#
# local pointwise branch
#
# ->
#
# nonlinear activation
#
# ============================================================

class WNOBlock(nn.Module):


    def __init__(
        self,
        channels,
        levels
    ):


        super().__init__()


        # ====================================================
        # Wavelet integral operator
        # ====================================================

        self.wavelet_operator = WaveletOperator2D(

            channels=channels,

            levels=levels
        )


        # ====================================================
        # Local pointwise operator
        # ====================================================

        self.local = nn.Conv2d(

            channels,

            channels,

            kernel_size=1
        )


        # ====================================================
        # Local spatial refinement
        # ====================================================

        self.refine = nn.Conv2d(

            channels,

            channels,

            kernel_size=3,

            padding=1
        )


        self.norm = nn.GroupNorm(

            num_groups=8,

            num_channels=channels
        )


    def forward(
        self,
        x
    ):


        # ====================================================
        # Global/multiscale wavelet branch
        # ====================================================

        x_wave = self.wavelet_operator(
            x
        )


        # ====================================================
        # Local branch
        # ====================================================

        x_local = self.local(
            x
        )


        # ====================================================
        # Combine
        # ====================================================

        x = (

            x_wave

            +

            x_local
        )


        x = self.norm(
            x
        )


        x = F.gelu(
            x
        )


        # ====================================================
        # Local refinement
        # ====================================================

        x_ref = self.refine(
            x
        )


        x = (

            x

            +

            x_ref
        )


        x = F.gelu(
            x
        )


        return x


# ============================================================
# WNO2D
#
# PT -> PA
# ============================================================

class WNO2D(nn.Module):


    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        width=48,
        n_layers=4,
        wavelet_levels=2
    ):


        super().__init__()


        self.width = width


        # ====================================================
        # Input:
        #
        # PT + normalized t coordinate + normalized x coordinate
        #
        # 1 + 2 = 3 channels
        # ====================================================

        self.lifting = nn.Conv2d(

            in_channels + 2,

            width,

            kernel_size=1
        )


        # ====================================================
        # WNO blocks
        # ====================================================

        self.blocks = nn.ModuleList(

            [

                WNOBlock(

                    channels=width,

                    levels=wavelet_levels
                )

                for _ in range(
                    n_layers
                )
            ]
        )


        # ====================================================
        # Projection
        # ====================================================

        self.proj1 = nn.Conv2d(

            width,

            128,

            kernel_size=1
        )


        self.proj2 = nn.Conv2d(

            128,

            out_channels,

            kernel_size=1
        )


    # ========================================================
    # Coordinate grid
    # ========================================================

    def get_grid(
        self,
        shape,
        device
    ):


        B, C, T, X = shape


        t = torch.linspace(

            0.0,

            1.0,

            T,

            device=device
        )


        x = torch.linspace(

            0.0,

            1.0,

            X,

            device=device
        )


        tt, xx = torch.meshgrid(

            t,

            x,

            indexing="ij"
        )


        grid = torch.stack(

            [
                tt,
                xx
            ],

            dim=0
        )


        grid = grid.unsqueeze(
            0
        )


        grid = grid.repeat(

            B,

            1,

            1,

            1
        )


        return grid


    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        x
    ):


        # ====================================================
        # Coordinate embedding
        # ====================================================

        grid = self.get_grid(

            x.shape,

            x.device
        )


        x = torch.cat(

            [
                x,
                grid
            ],

            dim=1
        )


        # ====================================================
        # Lifting
        # ====================================================

        x = self.lifting(
            x
        )


        # ====================================================
        # WNO layers
        # ====================================================

        for block in self.blocks:

            x = block(
                x
            )


        # ====================================================
        # Projection
        # ====================================================

        x = F.gelu(

            self.proj1(
                x
            )
        )


        x = self.proj2(
            x
        )


        return x


# ============================================================
# Build model
# ============================================================

model = WNO2D(

    in_channels=IN_CHANNELS,

    out_channels=OUT_CHANNELS,

    width=WIDTH,

    n_layers=N_LAYERS,

    wavelet_levels=WAVELET_LEVELS

).to(
    DEVICE
)


# ============================================================
# Shape check
# ============================================================

print(
    "\nChecking model dimensions..."
)


with torch.no_grad():


    test_input = torch.randn(

        1,

        1,

        NT,

        NX,

        device=DEVICE
    )


    test_output = model(
        test_input
    )


print(
    "Input shape :",
    test_input.shape
)


print(
    "Output shape:",
    test_output.shape
)


assert (

    test_output.shape

    ==

    test_input.shape

), \
    f"Output shape mismatch: {test_output.shape}"


del test_input

del test_output


if torch.cuda.is_available():

    torch.cuda.empty_cache()


# ============================================================
# Number of parameters
# ============================================================

n_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad
)


print(

    f"\nTrainable parameters: "
    f"{n_params:,}"
)


# ============================================================
# Relative L2
# ============================================================

def relative_l2(
    pred,
    target
):


    numerator = torch.norm(

        pred
        -
        target
    )


    denominator = (

        torch.norm(
            target
        )

        +

        1e-8
    )


    return (

        numerator
        /
        denominator
    )


# ============================================================
# Train one epoch
# ============================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):


    model.train()


    total_mse = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        # ====================================================
        # Forward
        # ====================================================

        pred = model(
            heat
        )


        # ====================================================
        # Same loss as FNO
        # ====================================================

        mse = F.mse_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        loss = (

            mse

            +

            0.1
            *
            rel
        )


        # ====================================================
        # Backpropagation
        # ====================================================

        optimizer.zero_grad(
            set_to_none=True
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            model.parameters(),

            max_norm=1.0
        )


        optimizer.step()


        # ====================================================
        # Statistics
        # ====================================================

        total_mse += mse.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Evaluation
# ============================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):


    model.eval()


    total_mse = 0.0

    total_mae = 0.0

    total_rel = 0.0


    for heat, acoustic in loader:


        heat = heat.to(

            DEVICE,

            non_blocking=True
        )


        acoustic = acoustic.to(

            DEVICE,

            non_blocking=True
        )


        pred = model(
            heat
        )


        mse = F.mse_loss(

            pred,

            acoustic
        )


        mae = F.l1_loss(

            pred,

            acoustic
        )


        rel = relative_l2(

            pred,

            acoustic
        )


        total_mse += mse.item()

        total_mae += mae.item()

        total_rel += rel.item()


    return (

        total_mse
        /
        len(loader),

        total_mae
        /
        len(loader),

        total_rel
        /
        len(loader)
    )


# ============================================================
# Plot loss
# ============================================================

def plot_loss(
    log_path
):


    data = np.loadtxt(

        log_path,

        delimiter=",",

        skiprows=1
    )


    if data.ndim == 1:

        data = data[
            None,
            :
        ]


    epoch = data[
        :,
        0
    ]


    train_mse = data[
        :,
        1
    ]


    train_rel = data[
        :,
        2
    ]


    val_mse = data[
        :,
        3
    ]


    val_rel = data[
        :,
        5
    ]


    # ========================================================
    # MSE
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_mse,

        label="Train MSE"
    )


    plt.semilogy(

        epoch,

        val_mse,

        label="Validation MSE"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "MSE"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            OUT_DIR,

            "loss_curve.png"
        ),

        dpi=300
    )


    plt.close()


    # ========================================================
    # Relative L2
    # ========================================================

    plt.figure(
        figsize=(5, 3.8)
    )


    plt.semilogy(

        epoch,

        train_rel,

        label="Train Rel. L2"
    )


    plt.semilogy(

        epoch,

        val_rel,

        label="Validation Rel. L2"
    )


    plt.xlabel(
        "Epoch"
    )


    plt.ylabel(
        "Relative L2"
    )


    plt.legend(
        frameon=False
    )


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            OUT_DIR,

            "relative_l2_curve.png"
        ),

        dpi=300
    )


    plt.close()


# ============================================================
# Plot prediction
# ============================================================

@torch.no_grad()
def plot_prediction(
    model,
    dataset,
    sample_index=0
):


    model.eval()


    PT, PA = dataset[
        sample_index
    ]


    # ========================================================
    # Predict
    # ========================================================

    PT_gpu = PT.unsqueeze(
        0
    ).to(
        DEVICE
    )


    pred = model(
        PT_gpu
    )


    pred = (

        pred
        .cpu()
        .squeeze(0)
        .squeeze(0)
        .numpy()
    )


    PT = (

        PT
        .squeeze(0)
        .numpy()
    )


    PA = (

        PA
        .squeeze(0)
        .numpy()
    )


    # ========================================================
    # Denormalization
    # ========================================================

    PT_real = dataset.denormalize_pt(
        PT
    )


    PA_real = dataset.denormalize_pa(
        PA
    )


    pred_real = dataset.denormalize_pa(
        pred
    )


    # ========================================================
    # Error
    # ========================================================

    err = (

        pred_real
        -
        PA_real
    )


    # ========================================================
    # Scale
    # ========================================================

    vmax = np.max(

        np.abs(
            PA_real
        )
    )


    vmax = max(
        vmax,
        1e-12
    )


    evmax = np.max(

        np.abs(
            err
        )
    )


    evmax = max(
        evmax,
        1e-12
    )


    # ========================================================
    # Plot
    # ========================================================

    plt.figure(
        figsize=(12, 3)
    )


    # --------------------------------------------------------
    # PT
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        1
    )


    plt.imshow(

        PT_real,

        aspect="auto",

        cmap="inferno"
    )


    plt.title(
        "Input PT"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # GT PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        2
    )


    plt.imshow(

        PA_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "GT PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Predicted PA
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        3
    )


    plt.imshow(

        pred_real,

        aspect="auto",

        cmap="seismic",

        vmin=-vmax,

        vmax=vmax
    )


    plt.title(
        "Pred PA"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    plt.subplot(
        1,
        4,
        4
    )


    plt.imshow(

        err,

        aspect="auto",

        cmap="seismic",

        vmin=-evmax,

        vmax=evmax
    )


    plt.title(
        "Error"
    )


    plt.xlabel(
        "x"
    )


    plt.ylabel(
        "t"
    )


    plt.colorbar()


    plt.tight_layout()


    plt.savefig(

        os.path.join(

            OUT_DIR,

            "prediction_comparison.png"
        ),

        dpi=300,

        bbox_inches="tight"
    )


    plt.close()


    # ========================================================
    # Save numerical prediction
    # ========================================================

    np.savez(

        os.path.join(

            OUT_DIR,

            "prediction_result.npz"
        ),

        PT=PT_real,

        PA=PA_real,

        PA_pred=pred_real,

        error=err
    )


# ============================================================
# Optimizer
# ============================================================

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=LR,

    weight_decay=WEIGHT_DECAY
)


# ============================================================
# Scheduler
# ============================================================

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(

    optimizer,

    T_max=EPOCHS
)


# ============================================================
# Training log
# ============================================================

log_path = os.path.join(

    OUT_DIR,

    "training_log.csv"
)


with open(

    log_path,

    "w",

    newline=""

) as f:


    writer = csv.writer(
        f
    )


    writer.writerow(

        [
            "epoch",
            "train_mse",
            "train_rel_l2",
            "val_mse",
            "val_mae",
            "val_rel_l2",
            "lr"
        ]
    )


# ============================================================
# Best model
# ============================================================

best_val = float(
    "inf"
)
# ============================================================
# Early stopping
# ============================================================



best_epoch = 0

best_model_path = os.path.join(

    OUT_DIR,

    "best_wno2d.pt"
)


# ============================================================
# Training loop
# ============================================================

epoch_bar = tqdm(

    range(
        1,
        EPOCHS + 1
    ),

    desc="Training",

    ncols=120
)


for epoch in epoch_bar:

    # ========================================================
    # Training
    # ========================================================

    train_mse, train_rel = train_one_epoch(
        model,
        train_loader,
        optimizer
    )

    # ========================================================
    # Validation
    # ========================================================

    (
        val_mse,
        val_mae,
        val_rel
    ) = evaluate(
        model,
        val_loader
    )

    # ========================================================
    # Scheduler
    # ========================================================

    scheduler.step()

    lr_now = optimizer.param_groups[0]["lr"]

    # ========================================================
    # Log
    # ========================================================

    with open(
        log_path,
        "a",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow(
            [
                epoch,
                train_mse,
                train_rel,
                val_mse,
                val_mae,
                val_rel,
                lr_now
            ]
        )

    # ========================================================
    # Save best model
    # ========================================================
    
    if val_rel < best_val:
    
        best_val = val_rel
        best_epoch = epoch
    
        torch.save(
            model.state_dict(),
            best_model_path
        )
    # ========================================================
    # Print
    # ========================================================

    if epoch == 1 or epoch % 10 == 0:

        print(
            f"\nEpoch {epoch:04d} | "
            f"Train MSE {train_mse:.4e} | "
            f"Train Rel {train_rel:.4e} | "
            f"Val MSE {val_mse:.4e} | "
            f"Val MAE {val_mae:.4e} | "
            f"Val Rel {val_rel:.4e} | "
        )

    epoch_bar.set_postfix(
        train_mse=f"{train_mse:.2e}",
        val_mse=f"{val_mse:.2e}",
        rel=f"{val_rel:.2e}",
        lr=f"{lr_now:.1e}",
    )


# ============================================================
# Load best model
# ============================================================

print(
    "\nLoading best model..."
)


model.load_state_dict(

    torch.load(

        best_model_path,

        map_location=DEVICE,

        weights_only=True
    )
)


# ============================================================
# Final test
# ============================================================

(
    test_mse,
    test_mae,
    test_rel

) = evaluate(

    model,

    test_loader
)


print(
    "\nFinal Test Results"
)


print(
    f"Test MSE     : {test_mse:.6e}"
)


print(
    f"Test MAE     : {test_mae:.6e}"
)


print(
    f"Test Rel L2  : {test_rel:.6e}"
)


# ============================================================
# Save test results
# ============================================================

with open(

    os.path.join(

        OUT_DIR,

        "test_results.txt"
    ),

    "w"

) as f:


    f.write(
        "Final Test Results\n"
    )


    f.write(

        f"Test MSE     : "
        f"{test_mse:.6e}\n"
    )


    f.write(

        f"Test MAE     : "
        f"{test_mae:.6e}\n"
    )


    f.write(

        f"Test Rel L2  : "
        f"{test_rel:.6e}\n"
    )


# ============================================================
# Plot
# ============================================================

plot_loss(
    log_path
)


plot_prediction(

    model,

    test_dataset,

    sample_index=0
)


# ============================================================
# Finish
# ============================================================

print(

    f"\nBest validation Rel L2: "
    f"{best_val:.6e}"
)


print(

    f"Best model saved to: "
    f"{best_model_path}"
)


print(

    f"\nAll results saved to: "
    f"{OUT_DIR}"
)

Results will be saved to:
wno2d_mat_dataset_results/run_20260826_121001
Using device: cuda
Total .mat samples found: 800

Dataset split:
Train samples: 640
Val samples  : 80
Test samples : 80

Calculating normalization statistics...


Statistics: 100%|█████████████████████████████████████████████████████████████████████| 640/640 [00:56<00:00, 11.28it/s]



Normalization statistics:
PT mean = 3.135567e+01
PT std  = 2.164740e+01
PA mean = -2.100887e+04
PA std  = 2.847824e+05

Batch check:
PT batch shape: torch.Size([4, 1, 501, 200])
PA batch shape: torch.Size([4, 1, 501, 200])

Checking model dimensions...
Input shape : torch.Size([1, 1, 501, 200])
Output shape: torch.Size([1, 1, 501, 200])

Trainable parameters: 174,785


Training:   0%|     | 1/1000 [01:21<22:32:31, 81.23s/it, lr=1.0e-03, rel=1.34e+00, train_mse=9.46e-01, val_mse=7.70e-01]


Epoch 0001 | Train MSE 9.4630e-01 | Train Rel 1.2325e+00 | Val MSE 7.6973e-01 | Val MAE 2.4018e-01 | Val Rel 1.3351e+00 | 


...

Training:  39%|▍| 390/1000 [7:28:17<12:58:37, 76.59s/it, lr=6.7e-04, rel=3.95e-01, train_mse=8.21e-02, val_mse=1.38e-01]


Epoch 0390 | Train MSE 8.2115e-02 | Train Rel 3.2331e-01 | Val MSE 1.3750e-01 | Val MAE 1.0449e-01 | Val Rel 3.9548e-01 | 


Training:  40%|▍| 400/1000 [7:41:02<12:43:27, 76.35s/it, lr=6.5e-04, rel=3.95e-01, train_mse=7.90e-02, val_mse=1.34e-01]


Epoch 0400 | Train MSE 7.8958e-02 | Train Rel 3.0644e-01 | Val MSE 1.3375e-01 | Val MAE 1.0266e-01 | Val Rel 3.9507e-01 | 


Training:  40%|▍| 401/1000 [7:42:18<12:39:01, 76.03s/it, lr=6.5e-04, rel=4.10e-01, train_mse=8.16e-02, val_mse=1.53e-01]